# Walker + Attention: exact temporal Top-K probe

This is the next diagnostic after dense temporal attention improved ML-1M. It loads the **best checkpoint from the currently running Walker+Attention experiment** and evaluates the exact same weights with only the top **4 / 8 / 16 / 32 / 64** historical attention keys retained per query/head.

Important: this still computes dense QK scores. It is an **oracle sparsity test**, not yet a speed optimization. If small K preserves NDCG, the next step is to replace the dense search for those keys with SWG/HNSW navigation.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, sys, subprocess, shutil, runpy
REPO='/content/Sparsewalker'
BRANCH='agent/walker-temporal-topk'
if os.path.exists(REPO): shutil.rmtree(REPO)
subprocess.run(['git','clone','-q','-b',BRANCH,'https://github.com/hanialshater/Sparsewalker-.git',REPO],check=True)
SRC=f'{REPO}/src'; EXP=f'{REPO}/experiments'
sys.path.insert(0,SRC); sys.path.insert(0,EXP)
for name in list(sys.modules):
    if name=='sparsewalker' or name.startswith('sparsewalker.'):
        del sys.modules[name]

import torch
assert torch.cuda.is_available(), 'GPU runtime required'
print('GPU',torch.cuda.get_device_name(0),'bf16',torch.cuda.is_bf16_supported(),flush=True)
print('BRANCH',BRANCH,flush=True)

## Run the same-checkpoint sparsity probe

The current dense-attention experiment saves its best checkpoint to `MyDrive/sparsewalker_attention_control/ml1m/seed42/best.pt`, so you can run this while the original training continues. Re-running later automatically probes whichever newer checkpoint is best at that time.

In [ ]:
SCRIPT=f'{REPO}/experiments/run_ml1m_temporal_topk_probe.py'
sys.argv=[SCRIPT,
          '--seed','42',
          '--eval-batch-size','1024',
          '--ks','4','8','16','32','64']
print('INPROCESS TEMPORAL TOPK PROBE START',flush=True)
runpy.run_path(SCRIPT,run_name='__main__')
print('INPROCESS TEMPORAL TOPK PROBE END',flush=True)

## Compact result

Paste `DENSE_REFERENCE`, `ATTENTION_MASS`, all `TOPK_EVAL` lines, and `DECISION` back into ChatGPT.

In [ ]:
import json
from pathlib import Path
p=Path('/content/drive/MyDrive/sparsewalker_temporal_topk/result.json')
if p.exists():
    r=json.loads(p.read_text())
    print('CHECKPOINT_EPOCH',r['checkpoint_epoch'])
    print('DENSE',json.dumps(r['dense'],indent=2))
    print('ATTENTION_MASS',json.dumps(r['attention_mass'],indent=2))
    print('TOPK',json.dumps(r['topk'],indent=2))
    print('DECISION',json.dumps(r['decision'],indent=2))
